# 01 — Earthquake Gutenberg-Richter b-value

**Goal.** Reproduce the global tectonic-earthquake b-value (~1.0) using one year of USGS catalog data and `soc-pipeline`.

**ETA.** ~5 minutes (mostly the USGS API pull).

**Source dataset.** USGS FDSN REST API. No private credentials. The published Phase 1 result of the structural-isomorphism project uses 5 years (2020-2025, M>=3.5, ~84k events). For this tutorial we pull 1 year (2020, M>=3.5, ~15k events) which is enough to recover b within ±0.05.

**Expected.** b in [0.95, 1.15], CI width < 0.1, alpha_equivalent ~ 1.67.

In [ ]:
# 1. Pull USGS catalog
import time
import numpy as np
import requests

URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "geojson",
    "starttime": "2020-01-01",
    "endtime": "2020-12-31",
    "minmagnitude": 3.5,
    "orderby": "time-asc",
}

feat = None
for attempt in range(3):
    try:
        r = requests.get(URL, params=params, timeout=180)
        r.raise_for_status()
        feat = r.json().get("features", [])
        print(f"USGS returned {len(feat)} events")
        break
    except Exception as e:
        print(f"attempt {attempt+1} failed: {e}")
        time.sleep(5)

if feat is None or len(feat) == 0:
    # Fallback: synthetic G-R catalog with b=1.0 (for offline reproducibility)
    rng = np.random.default_rng(42)
    mc = 4.0
    n_synth = 15000
    mags = mc + rng.exponential(scale=1.0 / (1.0 * np.log(10)), size=n_synth)
    print(f"USGS unreachable — using synthetic catalog with b=1.0, n={n_synth}")
else:
    mags = np.array([
        f["properties"]["mag"] for f in feat
        if f["properties"].get("type") == "earthquake"
        and f["properties"].get("mag") is not None
    ])
    print(f"kept {len(mags)} tectonic earthquakes with valid magnitude")

In [ ]:
# 2. Fit b-value with soc-pipeline
from soc_pipeline import fit_b_value, b_to_clauset_alpha

result = fit_b_value(mags, bootstrap=True, n_boot=200, seed=42)
print(f"b = {result.b:.3f} ± {result.sigma_b:.3f}")
print(f"Mc = {result.mc:.2f}")
print(f"n above Mc = {result.n_above_mc}")
print(f"95% CI: [{result.ci_low:.3f}, {result.ci_high:.3f}]")
print(f"alpha-equivalent on energy: {result.alpha_equivalent:.3f}")

In [ ]:
# 3. Visualize the frequency-magnitude curve
import matplotlib.pyplot as plt

bins = np.arange(mags.min(), mags.max() + 0.1, 0.1)
counts, edges = np.histogram(mags, bins=bins)
centers = (edges[:-1] + edges[1:]) / 2
cum_counts = np.cumsum(counts[::-1])[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(centers, cum_counts, "o", label="cumulative N(M >= m)")
m_fit = np.linspace(result.mc, mags.max(), 50)
n_at_mc = (mags >= result.mc).sum()
a = np.log10(n_at_mc) + result.b * result.mc
log_n_fit = a - result.b * m_fit
ax.semilogy(m_fit, 10 ** log_n_fit, "r-", label=f"b = {result.b:.3f}")
ax.axvline(result.mc, color="gray", linestyle="--", alpha=0.5, label=f"Mc = {result.mc:.2f}")
ax.set_xlabel("magnitude M")
ax.set_ylabel("N(M' >= M)")
ax.set_title("Gutenberg-Richter on 2020 USGS catalog")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Verdict
from soc_pipeline import verdict_from_alpha_band

predicted = (0.95, 1.15)  # narrow tectonic b band
literature = (0.8, 1.2)
if predicted[0] <= result.b <= predicted[1]:
    print(f"CONFIRMED — b={result.b:.3f} in predicted [{predicted[0]}, {predicted[1]}]")
elif literature[0] <= result.b <= literature[1]:
    print(f"CONFIRMED (literature band) — b={result.b:.3f}")
else:
    print(f"DEVIATING — b={result.b:.3f} outside both bands")

print(f"\nPaper headline (5-yr global): b ≈ 1.08")
print(f"Your result (1-yr): b ≈ {result.b:.3f}")